# 0 — Creating the Unified Dataset

This notebook merges the outputs of the **dimensional model** (EmoBank — VAD predictions) and the **componential model** (GoEmotions — categorical emotion predictions) with the **survey-level variables** from `LAST_VERSION_noFuture_FINAL.csv`.

The row-to-ID mapping is provided by `ID.csv`.

| Source | File | Rows | Key columns |
|--------|------|------|-------------|
| Dimensional model | `english_documents_part1_emobank.xlsx` | 2 487 | `text`, `V_predicted`, `A_predicted`, `D_predicted` |
| Componential model | `english_documents_part2_goemotions.xlsx` | 2 487 | `text`, `predicted_emotions`, 28 emotion columns |
| Row ↔ ID mapping | `ID.csv` | 2 487 | row index → `ID` |
| Survey data | `LAST_VERSION_noFuture_FINAL.csv` | 2 487 | `ID` + 79 survey variables |
| **Output** | `merged_BERT_survey.xlsx` | **2 487** | All columns merged on `ID` |

In [ ]:
import pandas as pd
import os
from pathlib import Path

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# Adjust BASE_DIR if you run this notebook from a different location
BASE_DIR = Path("../..")

PATH_DIMENSIONAL  = BASE_DIR / "scripts/dimensional_model/output/english_documents_part1_emobank.xlsx"
PATH_COMPONENTIAL = BASE_DIR / "scripts/componential_model/output/english_documents_part2_goemotions.xlsx"
PATH_IDS          = BASE_DIR / "scripts/analysis/dataset_diLiberatore/ID.csv"
PATH_SURVEY       = BASE_DIR / "scripts/analysis/dataset_diLiberatore/LAST_VERSION_noFuture_FINAL.csv"

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "merged_BERT_survey.xlsx"

print(f"Dimensional model file : {PATH_DIMENSIONAL}")
print(f"Componential model file: {PATH_COMPONENTIAL}")
print(f"ID mapping file        : {PATH_IDS}")
print(f"Survey data file       : {PATH_SURVEY}")
print(f"Output path            : {OUTPUT_PATH}")

## 2. Load the BERT model outputs

In [ ]:
df_dimensional  = pd.read_excel(PATH_DIMENSIONAL)
df_componential = pd.read_excel(PATH_COMPONENTIAL)

print(f"Dimensional  (EmoBank)    — shape: {df_dimensional.shape}")
print(f"Componential (GoEmotions) — shape: {df_componential.shape}")

In [ ]:
print("\n── Dimensional model columns ──")
print(df_dimensional.columns.tolist())

print("\n── Componential model columns ──")
print(df_componential.columns.tolist())

## 3. Sanity checks & combine BERT predictions

In [ ]:
# Verify that both model outputs are positionally aligned
assert len(df_dimensional) == len(df_componential), (
    f"Row count mismatch: dimensional={len(df_dimensional)}, componential={len(df_componential)}"
)
assert (df_dimensional["text"] == df_componential["text"]).all(), (
    "Text columns are not aligned between dimensional and componential outputs!"
)

print(f"✅ Both outputs have {len(df_dimensional)} rows with identical text alignment.")

In [ ]:
# Combine by position (drop duplicate 'text' column from componential)
df_bert = pd.concat(
    [df_dimensional, df_componential.drop(columns=["text"])],
    axis=1
)

print(f"Combined BERT predictions — shape: {df_bert.shape}")
print(f"Columns: {df_bert.columns.tolist()}")

## 4. Attach IDs via ID.csv

In [ ]:
df_ids = pd.read_csv(PATH_IDS)
print(f"ID mapping — shape: {df_ids.shape}")
print(f"Columns: {df_ids.columns.tolist()}")
print(f"ID range: {df_ids['x'].min()} – {df_ids['x'].max()}")

assert len(df_ids) == len(df_bert), (
    f"ID.csv has {len(df_ids)} rows but BERT predictions have {len(df_bert)}!"
)

df_bert["ID"] = df_ids["x"].values
print(f"\n✅ ID column added — BERT shape: {df_bert.shape}")

## 5. Load the survey data

In [ ]:
df_survey = pd.read_csv(PATH_SURVEY, encoding="latin-1")

# Drop the unnamed row-index column if present
if "Unnamed: 0" in df_survey.columns:
    df_survey = df_survey.drop(columns=["Unnamed: 0"])

print(f"Survey data — shape: {df_survey.shape}")
print(f"Unique IDs         : {df_survey['ID'].nunique()}")
print(f"ID range           : {df_survey['ID'].min()} – {df_survey['ID'].max()}")

## 6. Inner join on ID

In [ ]:
df_merged = pd.merge(df_bert, df_survey, on="ID", how="inner")

print(f"Merged dataset shape : {df_merged.shape}")
print(f"  BERT columns       : {len(df_bert.columns) - 1}")    # minus ID
print(f"  Survey columns     : {len(df_survey.columns) - 1}")  # minus ID
print(f"  Total (with ID)    : {len(df_merged.columns)}")
print(f"  Rows (inner join)  : {len(df_merged)}")

In [ ]:
# Quick look at the VAD summary statistics
vad_cols = ["V_predicted", "A_predicted", "D_predicted"]
df_merged[vad_cols].describe().round(3)

In [ ]:
df_merged.head()

## 7. Save the unified dataset

In [ ]:
df_merged.to_excel(OUTPUT_PATH, index=False)

size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"✅ Saved to: {OUTPUT_PATH}")
print(f"   Rows   : {len(df_merged)}")
print(f"   Columns: {len(df_merged.columns)}")
print(f"   Size   : {size_mb:.2f} MB")

In [ ]:
# Verify the saved file
df_check = pd.read_excel(OUTPUT_PATH)
assert df_check.shape == df_merged.shape, "Shape mismatch after re-reading!"
print(f"✅ Verification passed — re-loaded shape: {df_check.shape}")